In [157]:
import pandas as pd
df=pd.read_csv("Rajasthan_Heatwave.csv")
df.head()

,YEAR,MONTH,DAY,WIND_U10,WIND_V10,MSLP,BLH,GEOP,TEMP2M,TMAX,...,RAIN,SRAD,EVAP,SOILT1,SOILM1,LAI,HEATWAVE,LAT,LON,DISTRICT
0,2006,3,1,-0.223022,-0.440903,101036.375,2020.7317,1671.1079,304.75806,303.54395,...,0.0,3117056,-0.000023,312.75903,0.024887,0.515625,0,25.75,71.5,Barmer
1,2006,3,2,-1.344009,-0.294922,101076.440,2265.7910,1671.1079,306.45337,304.95703,...,0.0,3131200,-0.000024,312.66943,0.024345,0.515625,0,25.75,71.5,Barmer
2,2006,3,3,-0.960449,1.522965,101033.750,3152.6023,1671.1079,307.34863,306.21167,...,0.0,3105536,-0.000024,313.12280,0.023773,0.515625,0,25.75,71.5,Barmer
3,2006,3,4,2.704514,1.733017,100896.940,3146.9475,1671.1079,307.43457,306.70508,...,0.0,3126528,-0.000024,314.12085,0.023147,0.515625,0,25.75,71.5,Barmer
4,2006,3,5,2.717834,2.464432,101014.190,2069.5261,1671.1079,306.19507,304.88306,...,0.0,2776448,-0.000041,312.00806,0.027964,0.515625,0,25.75,71.5,Barmer


In [158]:
drop_columns=["YEAR","MONTH","DAY","TMAX","TMIN","LAT","LON","DISTRICT","HEATWAVE"]
X=df.drop(drop_columns,axis=1)
y=df["HEATWAVE"]

In [159]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42)


In [160]:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=100, random_state=42,max_depth=10)

In [161]:
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [162]:
import joblib
#joblib.dump(model, 'heatwave_model.pkl')

In [163]:
def predict_heatwave(input_data):
    # Load the trained model
    model = joblib.load('random_forest.pkl')
    
    # Convert input data to DataFrame
    input_df = input_data.copy()
    
    # Make prediction
    prediction = model.predict(input_df)[0]
    probability = model.predict_proba(input_df)[0][1]  # Probability of heatwave (class 1)
    
    return prediction,probability

In [164]:
climate_docs = [
    "Heatwaves occur when temperature and pressure conditions are extreme with low humidity.",
    "High MSLP and low BLH contribute to heatwave formation.",
    "Low soil moisture and high solar radiation increase surface heating.",
    "Reduced cloud cover increases ground temperature.",
    "Heatwaves are intensified by dry atmospheric conditions and weak winds.",
    "Urban areas experience stronger heatwaves due to heat island effect."
]

In [165]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

In [166]:
import os
from dotenv import load_dotenv
load_dotenv()
openai_api_key=os.getenv("OPENAI_API_KEY")
embeddings=OpenAIEmbeddings(openai_api_key=openai_api_key,model="text-embedding-ada-002")

In [167]:
vectorstore = FAISS.from_texts(climate_docs, embeddings)

In [168]:
retriever=vectorstore.as_retriever(search_kwargs={"k": 3})

In [169]:
def retriever_context(query):     
    retrieved_docs = retriever.invoke(query)
    context = " ".join([doc.page_content for doc in retrieved_docs])
    return context


In [170]:
def generate_response(context, prediction,probability):
    if prediction==1:
        label="Heatwave expected"
    else:
        label="No heatwave expected"
    return f"""
    Prediction:{label}
    print(f"Probability of heatwave: {probability:.2f}")
    climate Insights (RAG):
    {context}

    Explaination:
    The prediction is based on the Random Forest model trained on historical climate data. 
    The model considers various features such as temperature, pressure, humidity, and 
    other relevant factors to determine the likelihood of a heatwave occurring. 
    The retrieved climate insights provide additional context and information about the conditions that contribute to heatwave formation."""

In [171]:
def heatwave_rag_system(input_df):

    # 1. ML Prediction
    prediction, probability = predict_heatwave(input_df)

    # 2. Create query for retrieval
    query = f"""
    WIND_U10 {input_df['WIND_U10'].iloc[0]}
    WIND_V10 {input_df['WIND_V10'].iloc[0]}
    MSLP {input_df['MSLP'].iloc[0]}
    BLH {input_df['BLH'].iloc[0]}
    TEMP2M {input_df['TEMP2M'].iloc[0]}
    CLOUD {input_df['CLOUD'].iloc[0]}
    RAIN {input_df['RAIN'].iloc[0]}
    SRAD {input_df['SRAD'].iloc[0]}
    """

    # 3. Retrieve context
    context = retriever_context(query)

    # 4. Generate explanation
    output = generate_response(context, prediction, probability)

    return output

In [200]:
y_test

20060    0
8589     0
17105    0
12430    0
7152     0
        ..
2932     0
12930    0
17098    0
19574    0
2669     0
Name: HEATWAVE, Length: 4392, dtype: int64

In [195]:
sample_input = X_test.iloc[[509]][X.columns].copy()
#sample_input=sample_input[X.columns]

result = heatwave_rag_system(sample_input)
print(result)


    Prediction:No heatwave expected
    print(f"Probability of heatwave: 0.02")
    climate Insights (RAG):
    High MSLP and low BLH contribute to heatwave formation. Heatwaves are intensified by dry atmospheric conditions and weak winds. Reduced cloud cover increases ground temperature.

    Explaination:
    The prediction is based on the Random Forest model trained on historical climate data. 
    The model considers various features such as temperature, pressure, humidity, and 
    other relevant factors to determine the likelihood of a heatwave occurring. 
    The retrieved climate insights provide additional context and information about the conditions that contribute to heatwave formation.
